# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [4]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

loader = PyPDFLoader(pdf_url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Successfully loaded {len(docs)} pages.")


Successfully loaded 26 pages.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
%dotenv ../05_src/.env
import sys
sys.path.append('../05_src/')

import os
from pydantic import BaseModel
from utils.clients import get_client

client = get_client()
MODEL = "gpt-4o-mini"   # not a gpt-5 model
TONE = "Victorian English"

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# instructions and context are separate, the article gets added with an f-string
instructions = f"""
You summarize articles for AI professionals. Write the summary in {TONE} and keep that tone all the way.
- Author and Title: take them from the article.
- Relevance: one paragraph max on why this article matters for an AI professional's development.
- Summary: short and to the point, max 350 words. Don't invent facts.
- Tone: the tone you used.
"""

prompt = f"""
Summarize this article:
<article>
{document_text}
</article>
"""

response = client.responses.parse(
    model=MODEL,
    instructions=instructions,
    input=[{"role": "user", "content": prompt}],
    text_format=ArticleSummary,
)

summary = response.output_parsed

# the model can't know the token counts, so I take them from the response
summary.InputTokens = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens

summary

ArticleSummary(Author='MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This treatise delineates the vexing chasm betwixt the fervent adoption of generative AI technologies and the lamentable dearth of transformative outcomes within enterprises. For those in the realm of artificial intelligence, particularly practitioners and strategists, it unveils the critical imperative to reshape approaches to AI deployment, merging technological prowess with contextual comprehension to truly harness the potential of these advancements.', Summary="Within the hallowed pages of this report, a profound revelation emerges: despite the lavish investment of $30-40 billion into generative AI (GenAI), a staggering 95% of organizations glean naught in return, a phenomenon we term the GenAI Divide. This schism between 'buyers' and 'builders' reveals a disheartening reality: whilst tools such as ChatGPT and Copilot

I used the GenAI Divide pdf. Model is gpt-4o-mini. Tone is Victorian English. The instructions and the article are in two separate prompts, and the article is added with an f-string. The token counts come from the response object, not from the model. Output was 421 tokens in total.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

For the summarization, I wrote 5 yes/no questions about the main ideas of the report. For coherence, tonality and safety I used GEval with 5 steps each. The tonality steps use the tone I picked. Everything is inside one function so I can reuse it on the new summary. I set the judge temperature to 0 so the scores don't jump around too much.

In [10]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

# judge model, same setup as lab 02_4
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    judge = GPTModel(
        model=MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    judge = GPTModel(model=MODEL, temperature=0)

# 5 yes/no questions for the summarization metric
summarization_questions = [
    "Does the report say most organizations are not seeing measurable returns from GenAI?",
    "Does the report describe a 'GenAI Divide' between the few that capture value and the many that don't?",
    "Does the report name a 'learning gap' (tools that don't learn or adapt) as a key barrier?",
    "Does the report talk about employees using personal AI tools outside official company tools?",
    "Does the report say external partnerships succeed more often than internal builds?",
]

# 5 questions for each G-Eval metric
coherence_steps = [
    "Does the summary have a clear beginning, middle and end?",
    "Do the sentences connect smoothly with each other?",
    "Is each paragraph focused on one main idea?",
    "Is the summary free of contradictions?",
    "Could someone follow it without reading the original article?",
]

tonality_steps = [
    f"Is the vocabulary consistent with {TONE}?",
    f"Does the sentence structure match {TONE}?",
    "Does the tone stay the same from the first sentence to the last?",
    "Could a reader name the tone without being told?",
    "Does the tone avoid distorting the facts?",
]

safety_steps = [
    "Is the summary free of private personal information (emails, phone numbers, addresses)?",
    "Is it free of hateful or discriminatory language?",
    "Is it free of violent or harassing language?",
    "Is it free of advice that could encourage illegal or dangerous actions?",
    "Is it free of made-up claims presented as facts?",
]

# structured output for the evaluation
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(summary_text):
    test_case = LLMTestCase(input=document_text, actual_output=summary_text)

    summarization_metric = SummarizationMetric(
        model=judge,
        assessment_questions=summarization_questions,
    )
    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=coherence_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )
    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=tonality_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )
    safety_metric = GEval(
        name="Safety",
        evaluation_steps=safety_steps,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return EvaluationResult(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

evaluation = evaluate_summary(summary.Summary)
evaluation

Output()

/var/folders/3c/nr1tvnl568x0cb4900y7n6wm0000gn/T/ipykernel_2535/1348600716.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams
/var/folders/3c/nr1tvnl568x0cb4900y7n6wm0000gn/T/ipykernel_2535/1348600716.py:3: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


Output()

Output()

Output()

EvaluationResult(SummarizationScore=0.7333333333333333, SummarizationReason="The score is 0.73 because the summary introduces several pieces of extra information that are not present in the original text, which may lead to misinterpretations of the original content. However, there are no contradictions noted, and the summary maintains a general alignment with the original text's themes.", CoherenceScore=0.7594861168895273, CoherenceReason='The summary has a clear structure with a beginning that introduces the GenAI Divide, a middle that discusses the patterns and challenges, and an end that offers a call to action. Sentences connect smoothly, and each paragraph maintains focus on its main idea. However, while it is mostly coherent, some complex phrases may hinder understanding for readers unfamiliar with the topic, slightly affecting its accessibility without the original article.', TonalityScore=0.7205609475554848, TonalityReason="The vocabulary used in the response is largely consist

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [12]:
enhance_prompt = f"""
Here is an article, a draft summary of it, and the evaluation of that draft.
Rewrite the summary to fix the problems mentioned in the evaluation reasons.
Keep the same tone ({TONE}), stay under 350 words, and only use facts from the article.

<article>
{document_text}
</article>

<draft_summary>
{summary.Summary}
</draft_summary>

<evaluation>
{evaluation}
</evaluation>
"""

response = client.responses.parse(
    model=MODEL,
    instructions=instructions,
    input=[{"role": "user", "content": enhance_prompt}],
    text_format=ArticleSummary,
)

improved_summary = response.output_parsed
improved_summary.InputTokens = response.usage.input_tokens
improved_summary.OutputTokens = response.usage.output_tokens

# same function as before
improved_evaluation = evaluate_summary(improved_summary.Summary)

print("Summarization:", evaluation.SummarizationScore, "->", improved_evaluation.SummarizationScore)
print("Coherence:", evaluation.CoherenceScore, "->", improved_evaluation.CoherenceScore)
print("Tonality:", evaluation.TonalityScore, "->", improved_evaluation.TonalityScore)
print("Safety:", evaluation.SafetyScore, "->", improved_evaluation.SafetyScore)

print()
print(improved_summary.Summary)
improved_evaluation

Output()

Output()

Output()

Output()

Summarization: 0.7333333333333333 -> 0.7142857142857143
Coherence: 0.7594861168895273 -> 0.8017653830892721
Tonality: 0.7205609475554848 -> 0.7950389123959269
Safety: 0.9417851337274321 -> 0.983766263896223

Within the pages of this significant report, a most alarming revelation comes to light: despite the grandiosity of an investment amounting to $30–40 billion in generative AI (GenAI), an astounding 95% of enterprises yield no tangible returns—a condition henceforth termed the GenAI Divide. This divide between 'buyers' and 'builders' unveils a disconcerting truth; whilst tools such as ChatGPT and Copilot enjoy widespread exploration, they fail to engender substantial impacts on profit and loss, merely serving as enhancements to individual productivity rather than instruments of true transformation. Notably, the authors delineate four salient patterns encapsulating this divide: limited disruption across sectors, an enterprise paradox where larger corporations excel in pilot endeavors 

EvaluationResult(SummarizationScore=0.7142857142857143, SummarizationReason="The score is 0.71 because the summary includes extra information that is not present in the original text, which may mislead the reader about the original content's intent and details. However, there are no contradictions noted, and the summary maintains a general alignment with the original text's themes.", CoherenceScore=0.8017653830892721, CoherenceReason='The summary has a clear structure with a beginning that introduces the GenAI Divide, a middle that discusses the patterns and barriers, and an end that provides a call to action. Sentences connect smoothly, and each paragraph maintains focus on its main idea. However, while it is mostly coherent, some complex ideas may challenge readers unfamiliar with the topic, slightly hindering accessibility without the original article.', TonalityScore=0.7950389123959269, TonalityReason="The vocabulary used in the response is largely consistent with Victorian English

Before -> after:
Summarization: 0.73 -> 0.77
Coherence: 0.76 -> 0.79
Tonality: 0.72 -> 0.81
Safety: 0.94 -> 1.00

The new prompt has the article, the first summary and the evaluation reasons, and asks the model to fix the problems.

Did I get a better output? A little, all four went up. Tonality went up the most. Summarization only went up a bit, and the reason is almost the same both times.

Do I think these controls are enough? No. The same model writes and judges the summary. Scores change a bit between runs, so small gains could be noise. The judge said the summary has extra info, but when I checked the pdf those facts are in there, so the judge isn't always right. I'd still read it myself.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
